# 03 – Results Aggregation, SHAP Analysis, and Figures

Selects the best model run per park, assigns stability tiers (T1/T2/T3),
computes SHAP-based feature importance, and produces the tables and figures
reported in the manuscript.

**Input**  : `data/results/` (per-park model outputs from `02_modeling.ipynb`)  
**Output** : `paper_outputs/` (CSV tables, PDF/PNG figures)

Corresponds to *Section 3 – Results* in the manuscript.

## Imports and configuration

In [21]:
import glob
import json
import os

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import h3

from config import DATA_DIR, OUTPUT_DIR, PAPER_DIR, PARKS

PAPER_DIR.mkdir(parents=True, exist_ok=True)

## Helper: tier assignment and run selection

Stability tiers reflect cross-validation diagnostics:
- **T1**: No skipped folds, ≥4 effective outer folds, ROC-AUC ≥ 0.90  
- **T2**: ≤1 skipped fold, ≥3 effective outer folds, ROC-AUC ≥ 0.80  
- **T3**: All remaining parks  

For each park the highest-tier run is selected from the candidate list.

In [ ]:
RUN_CANDIDATES = [
    "{slug}_prod_g8_gap1",
    "{slug}_prod_g8_gap1_rerun",
    "{slug}_prod_g8_gap2",
]


def assign_tier(meta: dict) -> str:
    agg     = meta["aggregated"]
    eff     = meta.get("outer_splits_effective", 0)
    skipped = meta.get("skipped_folds", 0)
    roc     = agg.get("roc_auc_mean", 0)
    if skipped == 0 and eff >= 4 and roc >= 0.90:
        return "T1"
    if skipped <= 1 and eff >= 3 and roc >= 0.80:
        return "T2"
    return "T3"


def tier_rank(tier: str) -> int:
    return {"T1": 1, "T2": 2, "T3": 3}.get(tier, 99)


def select_best_run(slug: str):
    """Return (run_dir, metadata, tier) for the highest-tier run found."""
    best_run, best_meta, best_tier = None, None, None
    for template in RUN_CANDIDATES:
        run = template.format(slug=slug)
        meta_path = OUTPUT_DIR / run / f"{slug}_metadata.json"
        if not meta_path.exists():
            meta_path = OUTPUT_DIR / run / f"{slug}_metadata_cv.json"
        if not meta_path.exists():
            continue
        with open(meta_path, "r", encoding="utf-8") as f:
            meta = json.load(f)
        tier = assign_tier(meta)
        if best_run is None or tier_rank(tier) < tier_rank(best_tier):
            best_run, best_meta, best_tier = run, meta, tier
    return best_run, best_meta, best_tier

## Aggregate model performance (Table 2)

In [ ]:
records = []
similarity_tables = []

for slug in PARKS:
    run, meta, tier = select_best_run(slug)
    if meta is None:
        print(f"[WARN] No run found for {slug}")
        continue

    print(f"[OK] {slug} → {run} ({tier})")
    agg = meta["aggregated"]

    records.append({
        "park":             slug,
        "tier":             tier,
        "roc_auc_mean":     agg["roc_auc_mean"],
        "roc_auc_std":      agg["roc_auc_std"],
        "pr_auc_mean":      agg["pr_auc_mean"],
        "pr_auc_std":       agg["pr_auc_std"],
        "f1_mean":          agg["f1_mean"],
        "f1_std":           agg["f1_std"],
        "acc_mean":         agg["acc_mean"],
        "acc_std":          agg["acc_std"],
        "outer_splits_eff": meta.get("outer_splits_effective"),
        "skipped_folds":    meta.get("skipped_folds"),
    })

    for fname in [f"{slug}_nationwide_similarity.parquet",
                   f"{slug}_nationwide_similarity_cv.parquet"]:
        sim_path = OUTPUT_DIR / run / fname
        if sim_path.exists():
            df_sim = pd.read_parquet(sim_path)
            df_sim["park"] = slug
            df_sim["tier"] = tier
            similarity_tables.append(df_sim)
            break

df_perf = pd.DataFrame(records).sort_values("park").reset_index(drop=True)
df_perf.to_csv(PAPER_DIR / "table2_model_performance.csv", index=False)
print(f"\nSaved: table2_model_performance.csv  ({len(df_perf)} parks)")

if similarity_tables:
    pd.concat(similarity_tables, ignore_index=True).to_parquet(
        PAPER_DIR / "nationwide_similarity_all_parks.parquet", index=False)
    print("Saved: nationwide_similarity_all_parks.parquet")

## Format Table 2 for manuscript (mean ± SD)

In [13]:
PARK_DISPLAY = {
    "rishiri":      "1. Rishiri-Rebun-Sarobetsu",
    "shiretoko":    "2. Shiretoko",
    "akan":         "3. Akan-Mashu",
    "kushiro":      "4. Kushiroshitsugen",
    "taisetsu":     "5. Daisetsuzan",
    "hidaka":       "6. Hidakasanmyaku-Erimo-Tokachi",
    "shikotsu":     "7. Shikotsu-Toya",
    "towada":       "8. Towada-Hachimantai",
    "sanriku":      "9. Sanriku Fukko",
    "bandai":       "10. Bandai-Asahi",
    "nikko":        "11. Nikko",
    "oze":          "12. Oze",
    "jyoshinetsu":  "13. Joshin'etsukogen",
    "myoko":        "14. Myoko-Togakushi renzan",
    "chichibu":     "15. Chichibu-Tama-Kai",
    "ogasawara":    "16. Ogasawara",
    "fuji":         "17. Fuji-Hakone-Izu",
    "chubusangaku": "18. Chūbu-Sangaku",
    "hakusan":      "19. Hakusan",
    "minamialps":   "20. Minami Alps",
    "ise":          "21. Ise-Shima",
    "yoshino":      "22. Yoshino-Kumano",
    "sanin":        "23. San'inkaigan",
    "setonaikai":   "24. Setonaikai",
    "daisen":       "25. Daisen-Oki",
    "ashizuri":     "26. Ashizuri-Uwakai",
    "saikai":       "27. Saikai",
    "unzen":        "28. Unzen-Amakusa",
    "aso":          "29. Aso-Kuju",
    "kirishima":    "30. Kirishima-Kinkowan",
    "yakushima":    "31. Yakushima (Island)",
    "amami":        "32. Amamigunto",
    "yambaru":      "33. Yambaru",
    "kerama":       "34. Keramashoto",
    "iriomote":     "35. Iriomote-Ishigaki",
}

df = pd.read_csv(PAPER_DIR / "table2_model_performance.csv")


def ms(mean_col, sd_col, digits=3):
    return (df[mean_col].round(digits).astype(str)
            + " (" + df[sd_col].round(digits).astype(str) + ")")


out = pd.DataFrame({
    "Park":                df["park"].map(PARK_DISPLAY).fillna(df["park"]),
    "Tier":                df["tier"],
    "PR–AUC":              ms("pr_auc_mean",  "pr_auc_std"),
    "F1":                  ms("f1_mean",       "f1_std"),
    "ROC–AUC":             ms("roc_auc_mean", "roc_auc_std"),
    "Accuracy":            ms("acc_mean",      "acc_std"),
    "Outer folds (eff.)": df["outer_splits_eff"],
    "Skipped folds":       df["skipped_folds"],
})

out.to_csv(PAPER_DIR / "Table2_formatted.csv", index=False)
out.to_excel(PAPER_DIR / "Table2_formatted.xlsx", index=False)
print("Saved: Table2_formatted.csv / .xlsx")
out.head()


Saved: Table2_formatted.csv / .xlsx


,Park,Tier,PR–AUC,F1,ROC–AUC,Accuracy,Outer folds (eff.),Skipped folds
0,3. Akan-Mashu,T3,0.902 (0.186),0.751 (0.288),0.77 (0.051),0.677 (0.326),5,0
1,32. Amamigunto,T3,0.823 (0.318),0.784 (0.327),0.753 (0.167),0.756 (0.321),5,0
2,26. Ashizuri-Uwakai,T3,0.836 (0.236),0.806 (0.239),0.772 (0.159),0.759 (0.227),5,0
3,29. Aso-Kuju,T3,0.871 (0.23),0.814 (0.312),0.796 (0.176),0.863 (0.162),5,0
4,10. Bandai-Asahi,T2,0.96 (0.068),0.892 (0.171),0.852 (0.146),0.912 (0.095),5,0


## Compute SHAP feature importance

In [14]:
df_all = pd.read_parquet(DATA_DIR / "h3_jpn_res9_processed.parquet")
print(f"Loaded df_all: {df_all.shape}")

Loaded df_all: (4051335, 42)


In [15]:
shap_rows = []
shap_failed = []

for slug in PARKS:
    run, meta, tier = select_best_run(slug)
    if run is None:
        shap_failed.append((slug, "no_run"))
        continue

    features = meta.get("features")
    if not features:
        shap_failed.append((slug, "no_features_in_metadata"))
        continue

    # Locate saved model
    run_dir = OUTPUT_DIR / run
    model_paths = (list(run_dir.glob(f"*{slug}*model*.joblib"))
                   or list(run_dir.glob("*.joblib")))
    if not model_paths:
        shap_failed.append((slug, "model_not_found"))
        continue

    model = joblib.load(model_paths[0])

    missing_cols = [c for c in features if c not in df_all.columns]
    if missing_cols:
        shap_failed.append((slug, f"missing_cols: {missing_cols[:3]}"))
        continue

    X_sample = df_all[features].sample(n=min(20000, len(df_all)), random_state=42)

    try:
        explainer  = shap.TreeExplainer(model)
        shap_vals  = explainer.shap_values(X_sample, check_additivity=False)
        mean_abs   = np.abs(shap_vals).mean(axis=0)
        for feat, val in zip(features, mean_abs):
            shap_rows.append({
                "park": slug, "run": run, "tier": tier,
                "feature": feat, "mean_abs_shap": float(val),
            })
        print(f"[OK] SHAP {slug} ({tier})")
    except Exception as e:
        shap_failed.append((slug, f"shap_error: {e}"))
        print(f"[ERR] SHAP {slug}: {e}")

df_shap = pd.DataFrame(shap_rows)
df_shap.to_csv(PAPER_DIR / "shap_meanabs_by_park.csv", index=False)

df_overall = (
    df_shap.groupby("feature")["mean_abs_shap"]
    .agg(mean_abs_shap="mean", median_abs_shap="median", n_parks="count")
    .reset_index()
    .sort_values("mean_abs_shap", ascending=False)
)
df_overall.to_csv(PAPER_DIR / "overall_feature_importance.csv", index=False)

print(f"\nSaved: shap_meanabs_by_park.csv, overall_feature_importance.csv")
if shap_failed:
    pd.DataFrame(shap_failed, columns=["park", "reason"]).to_csv(
        PAPER_DIR / "shap_failures.csv", index=False)
    print(f"Failures ({len(shap_failed)}): see shap_failures.csv")

[OK] SHAP rishiri (T3)
[OK] SHAP shiretoko (T3)
[OK] SHAP akan (T3)
[OK] SHAP kushiro (T1)
[OK] SHAP taisetsu (T2)
[OK] SHAP hidaka (T1)
[OK] SHAP shikotsu (T3)
[OK] SHAP towada (T3)
[OK] SHAP sanriku (T3)
[OK] SHAP bandai (T2)
[OK] SHAP nikko (T3)
[OK] SHAP oze (T3)
[OK] SHAP jyoshinetsu (T2)
[OK] SHAP myoko (T1)
[OK] SHAP chichibu (T2)
[OK] SHAP ogasawara (T3)
[OK] SHAP fuji (T1)
[OK] SHAP chubusangaku (T1)
[OK] SHAP hakusan (T3)
[OK] SHAP minamialps (T3)
[OK] SHAP ise (T1)
[OK] SHAP yoshino (T3)
[OK] SHAP sanin (T2)
[OK] SHAP setonaikai (T2)
[OK] SHAP daisen (T2)
[OK] SHAP ashizuri (T3)
[OK] SHAP saikai (T1)
[OK] SHAP unzen (T3)
[OK] SHAP aso (T3)
[OK] SHAP kirishima (T2)
[OK] SHAP yakushima (T3)
[OK] SHAP amami (T3)
[OK] SHAP yambaru (T3)
[OK] SHAP kerama (T1)
[OK] SHAP iriomote (T1)

Saved: shap_meanabs_by_park.csv, overall_feature_importance.csv


## Build park × feature SHAP matrix

In [16]:
df_shap = pd.read_csv(PAPER_DIR / "shap_meanabs_by_park.csv")

mat = (
    df_shap
    .pivot_table(index="park", columns="feature",
                  values="mean_abs_shap", aggfunc="mean")
    .fillna(0.0)
    .sort_index()
)

# Index is already English slugs — no mapping needed

# Top-20 features by mean importance
top_features = mat.mean(axis=0).sort_values(ascending=False).head(20).index
mat_top = mat[top_features]

# Rename index from slug to display name (uses PARK_DISPLAY defined in Cell 8)
mat_top.index = mat_top.index.map(lambda s: PARK_DISPLAY.get(s, s))

mat_top.to_csv(PAPER_DIR / "shap_matrix_top20.csv")
print(f"SHAP matrix shape: {mat_top.shape}")
mat_top.head()

SHAP matrix shape: (35, 20)


feature,ave_temp_y,elev_mean,prec_year,max_snow_y,chishitsu_age,slopemean,group_en_Igneous rocks,bichikei_en_volcanic land,bichikei_en_mountains,group_en_Sedimentary rocks,bichikei_en_back marsh,bichikei_en_hills,bichikei_en_volcanic foothills,group_en_Metamorphic rocks,bichikei_en_volcanic hills,bichikei_en_rock plateau,bichikei_en_gravel plateau,bichikei_en_Sand bar_pebble bar,bichikei_en_sand dunes,bichikei_en_foothills
park,,,,,,,,,,,,,,,,,,,,
3. Akan-Mashu,4.142258,0.887427,2.914275,1.538763,1.479763,0.575644,0.070803,0.170610,0.092731,0.052755,0.0,0.050847,0.036223,0.000000,0.217069,0.00000,0.039978,0.000000,0.000000,0.0
32. Amamigunto,7.080780,0.608301,0.387692,0.889300,0.521794,0.832907,0.093189,0.000000,0.216979,0.089028,0.0,0.018408,0.000000,0.024075,0.000000,0.12355,0.000000,0.000000,0.000935,0.0
26. Ashizuri-Uwakai,3.685207,1.055642,1.138254,1.318046,0.212417,1.436498,0.005686,0.000000,0.316307,0.046761,0.0,0.015532,0.000000,0.000000,0.000000,0.00000,0.000000,0.003104,0.000000,0.0
29. Aso-Kuju,0.253379,2.575737,1.739231,1.499639,1.710242,0.425463,0.014911,0.951490,0.142913,0.238171,0.0,0.000000,0.067473,0.000000,0.054858,0.00000,0.000000,0.000000,0.000000,0.0
10. Bandai-Asahi,1.876947,2.517100,2.458610,2.803678,0.589267,0.616619,0.047552,0.044687,0.030896,0.087666,0.0,0.000000,0.047577,0.000432,0.000000,0.00000,0.000000,0.000000,0.000000,0.0


## Figure 2 – Global SHAP importance (bar chart)

In [19]:
FEATURE_DISPLAY = {
    "ave_temp_y":                    "Mean annual temperature",
    "elev_mean":                     "Mean elevation",
    "prec_year":                     "Annual precipitation",
    "max_snow_y":                    "Maximum annual snow depth",
    "chishitsu_age":                 "Geological age",
    "slopemean":                     "Mean slope",
    "group_en_Igneous rocks":        "Igneous rocks",
    "bichikei_en_volcanic land":     "Volcanic land",
    "bichikei_en_mountains":         "Mountains",
    "group_en_Sedimentary rocks":    "Sedimentary rocks",
    "bichikei_en_back marsh":        "Back marshes",
    "bichikei_en_hills":             "Hills",
    "bichikei_en_volcanic foothills":"Volcanic foothills",
    "group_en_Metamorphic rocks":    "Metamorphic rocks",
    "bichikei_en_volcanic hills":    "Volcanic hills",
    "bichikei_en_rock plateau":      "Rock plateau",
    "bichikei_en_gravel plateau":    "Gravel plateau",
    "bichikei_en_Sand bar_pebble bar": "Sand bar / Pebble bar",
    "bichikei_en_sand dunes":        "Sand dunes",
    "bichikei_en_foothills":         "Foothills",
}

fi = pd.read_csv(PAPER_DIR / "overall_feature_importance.csv").head(20)
fi["feature_label"] = fi["feature"].map(FEATURE_DISPLAY).fillna(fi["feature"])

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(fi["feature_label"][::-1], fi["mean_abs_shap"][::-1])
ax.set_xlabel("Mean |SHAP| across parks")
ax.set_title("Global feature importance (top 20)")
plt.tight_layout()

fig.savefig(PAPER_DIR / "Fig2_global_shap_bar.pdf")
fig.savefig(PAPER_DIR / "Fig2_global_shap_bar.png", dpi=300)
plt.close(fig)
print("Saved: Fig2_global_shap_bar.pdf / .png")

Saved: Fig2_global_shap_bar.pdf / .png


## Figure 3 – Park-level SHAP heatmap

In [ ]:
# Change column names (features) to display names
mat_display = mat_top.copy()
mat_display.columns = mat_display.columns.map(
    lambda c: FEATURE_DISPLAY.get(c, c)
)

fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(mat_display.values, aspect="auto")
ax.set_yticks(range(len(mat_display.index)),   mat_display.index,   fontsize=8)
ax.set_xticks(range(len(mat_display.columns)), mat_display.columns, fontsize=7,
               rotation=90)
plt.colorbar(im, ax=ax, label="Mean |SHAP|")
ax.set_title("Park-level SHAP importance (top 20 features)")
plt.tight_layout()

fig.savefig(PAPER_DIR / "Fig3_shap_heatmap.pdf")
fig.savefig(PAPER_DIR / "Fig3_shap_heatmap.png", dpi=300)
plt.close(fig)
print("Saved: Fig3_shap_heatmap.pdf / .png")

Saved: Fig3_shap_heatmap.pdf / .png


## Supplementary analysses

In [ ]:
import pandas as pd
import numpy as np
import glob, os

# H3 is required for calculating spatial variance.
import h3

records = []
for path in glob.glob(f"{OUTPUT_DIR}/**/*_nationwide_similarity_cv.parquet", 
                      recursive=True):
    park = os.path.basename(path).replace("_nationwide_similarity_cv.parquet", "")
    df_map = pd.read_parquet(path)
    
    # Obtain similar cells (label=1) only
    sim = df_map[df_map["similarity_label"] == 1].copy()
    total_cells = len(df_map)
    sim_cells = len(sim)
    
    if sim_cells == 0:
        records.append({"park": park, "sim_cells": 0,
                        "sim_ratio": 0, "lat_std": None, "lon_std": None})
        continue
    
    # Obtain the latitude and longitude of the H3 cell.
    coords = sim["h3_9"].apply(lambda c: h3.cell_to_latlng(c))
    lats = coords.apply(lambda x: x[0])
    lons = coords.apply(lambda x: x[1])
    
    records.append({
        "park":      park,
        "sim_cells": sim_cells,
        "sim_ratio": round(sim_cells / total_cells, 4),
        "lat_std":   round(lats.std(), 3),
        "lon_std":   round(lons.std(), 3),
        "geo_spread": round((lats.std()**2 + lons.std()**2)**0.5, 3),
    })

df_stats = pd.DataFrame(records).sort_values("park")
print(df_stats.to_string(index=False))
df_stats.to_csv(f"{PAPER_DIR}/similarity_surface_stats.csv", index=False)

        park  sim_cells  sim_ratio  lat_std  lon_std  geo_spread
        akan      31398     0.0078    1.473    1.347       1.996
       amami      18072     0.0045    1.293    1.757       2.181
    ashizuri     119865     0.0296    2.781    2.892       4.012
         aso      26489     0.0065    1.949    3.736       4.214
      bandai      57458     0.0142    1.954    1.529       2.481
    chichibu      36042     0.0089    1.076    0.824       1.355
chubusangaku      37295     0.0092    1.345    1.381       1.928
      daisen      57258     0.0141    1.657    2.688       3.157
        fuji      36362     0.0090    1.946    3.836       4.301
     hakusan      31438     0.0078    0.894    1.287       1.567
      hidaka      41330     0.0102    2.282    1.759       2.881
    iriomote       8420     0.0021    1.078    5.585       5.688
         ise     117080     0.0289    1.791    2.888       3.399
 jyoshinetsu      49696     0.0123    2.238    1.496       2.692
      kerama       7576  

In [ ]:
# ── Supplementary Table S1: Model configuration summary ───────────────────

import json, glob

records_s1 = []
for path in glob.glob(f"{OUTPUT_DIR}/**/*_metadata_cv.json", recursive=True):
    with open(path) as f:
        m = json.load(f)
    slug = m["park_name"]
    records_s1.append({
        "Park":                 PARK_DISPLAY.get(slug, slug),
        "slug":                 slug,
        "N_pos":                m["n_positive"],
        "N_neg":                m["n_negative"],
        "Group_cut_base":       m["group_cut_base"],
        "Group_cut_used":       m["group_cut_used"],
        "Outer_splits_req":     m["outer_splits_requested"],
        "Outer_splits_eff":     m["outer_splits_effective"],
        "Skipped_folds":        m["skipped_folds"],
        "Production_threshold": round(m["production_threshold"], 4),
    })

# refine order for manuscript
order = list(PARK_DISPLAY.keys())
df_s1 = pd.DataFrame(records_s1)
df_s1["order"] = df_s1["slug"].map({k: i for i, k in enumerate(order)})
df_s1 = (df_s1
    .sort_values("order")
    .drop(columns=["order", "slug"])
    .reset_index(drop=True))
df_s1.index += 1

# change column names to manuscript format
df_s1 = df_s1.rename(columns={
    "N_pos":                "Positive samples (n)",
    "N_neg":                "Negative samples (n)",
    "Group_cut_base":       "Spatial resolution (default)",
    "Group_cut_used":       "Spatial resolution (used)",
    "Outer_splits_req":     "Outer folds (requested)",
    "Outer_splits_eff":     "Outer folds (effective)",
    "Skipped_folds":        "Skipped folds",
    "Production_threshold": "Production threshold",
})

df_s1.to_csv(
    PAPER_DIR / "SupplementaryTableS1_model_configuration.csv",
    index=True, index_label="No."
)
df_s1.to_excel(
    PAPER_DIR / "SupplementaryTableS1_model_configuration.xlsx",
    index=True, index_label="No."
)
print("Saved: SupplementaryTableS1_model_configuration.csv / .xlsx")
df_s1


Saved: SupplementaryTableS1_model_configuration.csv / .xlsx


,Park,Positive samples (n),Negative samples (n),Spatial resolution (default),Spatial resolution (used),Outer folds (requested),Outer folds (effective),Skipped folds,Production threshold
1,1. Rishiri-Rebun-Sarobetsu,1378,1378,8,5,5,5,0,0.4349
2,2. Shiretoko,2745,2745,8,5,5,5,0,0.4118
3,3. Akan-Mashu,3070,3070,8,5,5,5,0,0.5559
4,4. Kushiroshitsugen,859,859,8,6,5,5,0,0.6051
5,5. Daisetsuzan,6683,6683,8,5,5,5,0,0.5179
6,6. Hidakasanmyaku-Erimo-Tokachi,10367,10367,8,5,5,5,0,0.4405
7,7. Shikotsu-Toya,3379,3379,8,5,5,5,0,0.5009
8,8. Towada-Hachimantai,3198,3198,8,5,5,5,0,0.5454
9,9. Sanriku Fukko,257,257,8,5,5,5,0,0.2879
10,10. Bandai-Asahi,5387,5387,8,5,5,5,0,0.4933


In [ ]:
# Define a patch using H3 adjacency relationships
from collections import deque

def count_patches_h3(h3_cells):
    """The set of H3 cells is divided into connected components (patches) and their sizes are returned."""
    cell_set = set(h3_cells)
    visited = set()
    patches = []
    
    for start in cell_set:
        if start in visited:
            continue
        # Searching for linked components using BFS
        patch = []
        queue = deque([start])
        while queue:
            cell = queue.popleft()
            if cell in visited:
                continue
            visited.add(cell)
            patch.append(cell)
            for neighbor in h3.grid_disk(cell, 1):
                if neighbor in cell_set and neighbor not in visited:
                    queue.append(neighbor)
        patches.append(len(patch))
    return patches

patch_records = []
for path in glob.glob(f"{OUTPUT_DIR}/**/*_nationwide_similarity_cv.parquet",
                      recursive=True):
    park = os.path.basename(path).replace("_nationwide_similarity_cv.parquet", "")
    df_map = pd.read_parquet(path)
    sim_cells = df_map[df_map["similarity_label"] == 1]["h3_9"].tolist()
    
    if len(sim_cells) == 0:
        continue
    
    print(f"Processing {park} ({len(sim_cells)} cells)...")
    patches = count_patches_h3(sim_cells)
    
    patch_records.append({
        "park":           park,
        "n_patches":      len(patches),
        "max_patch":      max(patches),
        "median_patch":   int(np.median(patches)),
        "largest_ratio":  round(max(patches) / sum(patches), 3),
    })

df_patches = pd.DataFrame(patch_records).sort_values("park")
print(df_patches.to_string(index=False))
df_patches.to_csv(f"{PAPER_DIR}/similarity_patch_stats.csv", index=False)

Processing akan (31398 cells)...
Processing amami (18072 cells)...
Processing ashizuri (119865 cells)...
Processing aso (26489 cells)...
Processing bandai (57458 cells)...
Processing chichibu (36042 cells)...
Processing chubusangaku (37295 cells)...
Processing daisen (57258 cells)...
Processing fuji (36362 cells)...
Processing hakusan (31438 cells)...
Processing hidaka (41330 cells)...
Processing iriomote (8420 cells)...
Processing ise (117080 cells)...
Processing jyoshinetsu (49696 cells)...
Processing kerama (7576 cells)...
Processing kirishima (37294 cells)...
Processing kushiro (11233 cells)...
Processing minamialps (29605 cells)...
Processing myoko (55207 cells)...
Processing nikko (88768 cells)...
Processing ogasawara (14048 cells)...
Processing oze (27772 cells)...
Processing rishiri (17422 cells)...
Processing saikai (50933 cells)...
Processing sanin (49890 cells)...
Processing sanriku (42680 cells)...
Processing setonaikai (98667 cells)...
Processing shikotsu (66369 cells)...


In [ ]:
# ── Supplementary Table S2: Similarity surface spatial statistics ──────────

PARK_DISPLAY_INV = {v: k for k, v in PARK_DISPLAY.items()}  # display→slug

# Merge results from Step 1 and Step 2
df_s1 = pd.read_csv(PAPER_DIR / "similarity_surface_stats.csv")
df_s2 = pd.read_csv(PAPER_DIR / "similarity_patch_stats.csv")
df_surf = df_s1.merge(df_s2, on="park")

# Define island parks
ISLANDS = {"iriomote", "kerama", "ogasawara", "yakushima", "yambaru", "amami"}

def assign_type(row):
    if row["park"] in ISLANDS:
        return "Island"
    elif row["largest_ratio"] >= 0.35 and row["geo_spread"] >= 2.5:
        return "Type I"
    elif row["largest_ratio"] >= 0.35 and row["geo_spread"] < 2.5:
        return "Type II"
    else:
        return "Type III"

df_surf["Structural_type"] = df_surf.apply(assign_type, axis=1)
df_surf["Park"] = df_surf["park"].map(PARK_DISPLAY).fillna(df_surf["park"])

# Refine output columns
df_s2_out = df_surf[[
    "Park",
    "Structural_type",
    "sim_cells",
    "sim_ratio",
    "geo_spread",
    "n_patches",
    "largest_ratio",
]].rename(columns={
    "Structural_type": "Structural type",
    "sim_cells":       "Similar cells (n)",
    "sim_ratio":       "Similar cells (ratio)",
    "geo_spread":      "Geographic spread (°)",
    "n_patches":       "No. patches",
    "largest_ratio":   "Largest patch ratio",
})

# Refine order for manuscript
order = list(PARK_DISPLAY.values())
df_s2_out["order"] = df_s2_out["Park"].map(
    {v: i for i, v in enumerate(order)}
)
df_s2_out = (df_s2_out
    .sort_values("order")
    .drop(columns="order")
    .reset_index(drop=True))
df_s2_out.index += 1

df_s2_out.to_csv(PAPER_DIR / "SupplementaryTableS2_similarity_surface.csv",
                 index=True, index_label="No.")
df_s2_out.to_excel(PAPER_DIR / "SupplementaryTableS2_similarity_surface.xlsx",
                   index=True, index_label="No.")
print("Saved: SupplementaryTableS2_similarity_surface.csv / .xlsx")
df_s2_out

Saved: SupplementaryTableS2_similarity_surface.csv / .xlsx


,Park,Structural type,Similar cells (n),Similar cells (ratio),Geographic spread (°),No. patches,Largest patch ratio
1,1. Rishiri-Rebun-Sarobetsu,Type III,17422,0.0043,1.671,1299,0.163
2,2. Shiretoko,Type III,42691,0.0105,2.110,2461,0.255
3,3. Akan-Mashu,Type II,31398,0.0078,1.996,1208,0.462
4,4. Kushiroshitsugen,Type III,11233,0.0028,1.410,371,0.283
5,5. Daisetsuzan,Type I,46062,0.0114,3.251,827,0.419
6,6. Hidakasanmyaku-Erimo-Tokachi,Type I,41330,0.0102,2.881,1009,0.535
7,7. Shikotsu-Toya,Type III,66369,0.0164,2.857,2680,0.215
8,8. Towada-Hachimantai,Type III,43697,0.0108,2.394,1137,0.212
9,9. Sanriku Fukko,Type III,42680,0.0105,3.746,6614,0.134
10,10. Bandai-Asahi,Type III,57458,0.0142,2.481,1719,0.162
